# 🏗️ Chapter 10-03b: Take-Home Project & Portfolio

---

## 📖 The Real Test

Many GenAI interviews include a **take-home project**. They send you a brief on Friday, you return a working solution by Monday. This notebook shows you exactly how to approach it.

---

## 🎯 Common Take-Home Formats

```
╔══════════════════════════════════════════════════════════════════════╗
║  1. "Build a RAG chatbot over these PDFs"           (2-3 days)    ║
║  2. "Create an agent that can query our database"   (2-3 days)    ║
║  3. "Design and implement a content moderation pipe" (1-2 days)   ║
║  4. "Evaluate and improve this RAG pipeline"        (1-2 days)    ║
║  5. "Build a multi-step workflow with LangGraph"    (2-3 days)    ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 🧠 The Winning Structure

Every take-home should have this folder structure:

```
project/
├── README.md              ← Overview, setup, architecture diagram
├── requirements.txt       ← Pinned dependencies
├── .env.example           ← Template for API keys (never commit .env!)
├── src/
│   ├── __init__.py
│   ├── chain.py           ← Core LangChain/LangGraph logic
│   ├── tools.py           ← Tool definitions
│   ├── guardrails.py      ← Input/output validation
│   └── config.py          ← Settings and constants
├── app.py                 ← FastAPI or Streamlit entry point
├── tests/
│   ├── test_chain.py
│   └── test_guardrails.py
├── notebooks/
│   └── exploration.ipynb   ← Your experimentation notebook
└── docs/
    └── architecture.md     ← Design decisions explained
```

---

## 🧪 Step 1: The README Template

Your README is the FIRST thing reviewers see. Make it count.

In [ ]:
README_TEMPLATE = '''
# 🤖 Project Name

## Overview
One-paragraph description of what this does and why.

## Architecture
```
User Query → [Guardrails] → [Router] → [RAG/Agent] → [Output Guard] → Response
```

## Tech Stack
- **LLM**: GPT-4o-mini via LangChain
- **Vector DB**: ChromaDB
- **Framework**: LangGraph for agent orchestration
- **API**: FastAPI with streaming
- **Monitoring**: LangSmith tracing

## Quick Start
```bash
pip install -r requirements.txt
cp .env.example .env  # Add your API keys
python app.py
```

## Design Decisions
1. **Why ChromaDB**: Prototyping speed, easy metadata filtering
2. **Why LangGraph over LCEL**: Need agent loops and checkpointing
3. **Chunk size 800**: Balanced retrieval precision vs context

## Trade-offs & Future Work
- [ ] Add hybrid search (BM25 + vector)
- [ ] Replace InMemoryCache with Redis for persistence
- [ ] Add RAGAS evaluation pipeline

## Cost Estimation
~$0.002 per query (GPT-4o-mini + embeddings)
'''

print(README_TEMPLATE)

## 🧪 Step 2: Sample Take-Home — RAG Chatbot

In [ ]:
# Complete take-home solution skeleton

CHAIN_PY = '''
"""Core RAG chain with guardrails."""
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# Config
COLLECTION_NAME = "project_docs"
PERSIST_DIR = "./chroma_db"

# Components
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Prompt
RAG_PROMPT = ChatPromptTemplate.from_template("""
Answer based on the context below. If unsure, say "I don\'t know."

Context: {context}
Question: {question}
""")

def format_docs(docs):
    return "\\n\\n".join(d.page_content for d in docs)

# RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

def query(question: str) -> str:
    return rag_chain.invoke(question)
'''

print("📄 src/chain.py:")
print(CHAIN_PY)

In [ ]:
APP_PY = '''
"""FastAPI endpoint for RAG chatbot."""
from fastapi import FastAPI
from pydantic import BaseModel
from src.chain import query

app = FastAPI(title="RAG Chatbot")

class Question(BaseModel):
    text: str

@app.post("/ask")
async def ask(q: Question):
    answer = query(q.text)
    return {"answer": answer}

@app.get("/health")
async def health():
    return {"status": "ok"}
'''

print("📄 app.py:")
print(APP_PY)

In [ ]:
TEST_PY = '''
"""Tests for RAG chain."""
import pytest
from src.chain import query, format_docs
from langchain_core.documents import Document

def test_format_docs():
    docs = [
        Document(page_content="Hello world"),
        Document(page_content="Foo bar"),
    ]
    result = format_docs(docs)
    assert "Hello world" in result
    assert "Foo bar" in result

def test_query_returns_string():
    result = query("What is this project about?")
    assert isinstance(result, str)
    assert len(result) > 0
'''

print("📄 tests/test_chain.py:")
print(TEST_PY)

## 🧠 What Reviewers Look For

```
┌────────────────────────────────────────────────────────────────────┐
│  ✅ IMPRESSIVE                     │  ❌ RED FLAGS                │
├────────────────────────────────────────────────────────────────────┤
│  Clean project structure           │  All code in one file       │
│  README with architecture diagram  │  No README                  │
│  .env.example (no secrets!)        │  API keys in code           │
│  Error handling                    │  Crashes on bad input       │
│  Tests (even basic)                │  Zero tests                 │
│  Design decisions documented       │  No explanation of choices  │
│  Cost estimation                   │  Ignoring cost entirely     │
│  Guardrails included               │  No input validation        │
│  Trade-offs acknowledged           │  "My solution is perfect"   │
│  Streaming support                 │  Only sync responses        │
└────────────────────────────────────────────────────────────────────┘
```

---

## 🧠 Portfolio Projects That Stand Out

```
╔══════════════════════════════════════════════════════════════════════╗
║  PROJECT IDEAS FOR YOUR GITHUB PORTFOLIO                          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  1. 📚 RAG Knowledge Base                                         ║
║     Upload PDFs → chat with them → cite sources                   ║
║     Shows: RAG, embeddings, vector stores, LCEL                   ║
║                                                                    ║
║  2. 🔬 Research Agent                                             ║
║     Multi-step research with web search + analysis                ║
║     Shows: LangGraph, tool calling, multi-agent                   ║
║                                                                    ║
║  3. 🛡️ Content Moderator                                          ║
║     Classify + filter user content with safety layers             ║
║     Shows: Guardrails, classification, PII handling               ║
║                                                                    ║
║  4. 💬 Customer Support Bot                                       ║
║     RAG + tool calling + escalation + conversation memory         ║
║     Shows: Full-stack GenAI, production readiness                 ║
║                                                                    ║
║  5. 📊 Data Analyst Agent                                         ║
║     Natural language → SQL → visualization                        ║
║     Shows: Tool calling, structured output, data skills           ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 🧪 Step 3: Mock Interview Simulation

In [ ]:
# Self-assessment quiz — rate yourself 1-5 on each area

areas = {
    "Prompt Engineering (CoT, ReAct, Structured Output)": 0,
    "LangChain LCEL (pipes, runnables, parsers)": 0,
    "Embeddings & Vector Stores (cosine sim, ChromaDB)": 0,
    "RAG Pipeline (basic + advanced techniques)": 0,
    "Agents & Tool Calling (ReAct, function calling)": 0,
    "LangGraph (state, nodes, edges, checkpointing)": 0,
    "Multi-Agent (supervisor, swarm, CrewAI)": 0,
    "Fine-Tuning (LoRA, QLoRA, decision tree)": 0,
    "Production (caching, monitoring, deployment)": 0,
    "Safety (guardrails, PII, prompt injection)": 0,
}

print("📋 SELF-ASSESSMENT: Rate yourself 1-5 on each area")
print("   1=Beginner  3=Can explain  5=Can build from scratch\n")
for area in areas:
    print(f"  [ ] {area}")

print("\n💡 Focus your final review on any areas rated < 3")
print("   Re-read the concept notebooks for those chapters")

## ✅ Key Takeaways

```
┌───────────────────────────────────────────────────────────────────┐
│  📌 Take-homes: Clean structure > clever code. README is king.   │
├───────────────────────────────────────────────────────────────────┤
│  📌 Always include: tests, .env.example, architecture docs.     │
├───────────────────────────────────────────────────────────────────┤
│  📌 Document trade-offs — shows maturity and awareness.         │
├───────────────────────────────────────────────────────────────────┤
│  📌 Build 2-3 portfolio projects covering RAG, agents, safety.  │
├───────────────────────────────────────────────────────────────────┤
│  📌 Practice explaining concepts in 30 seconds — clarity wins.  │
└───────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Capstone Projects!

---

### 🎓 You're Ready to Ship!
Take-home template + interview prep + portfolio ideas — go get that GenAI role! 🚀